<a href="https://colab.research.google.com/github/Aya-KB/lora-moroccan-ornaments/blob/main/notebooks/00_donnees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 — Constitution du mini-jeu de données
Objectif : 20 à 50 images **autorisées** (Wikimedia Commons, photos de l'équipe), d'un **seul domaine** (zellige marocain), recadrées en 512×512, sans doublons, avec 5 à 10 images de test.

**Ordre :** `download` → relire/corriger les légendes dans `data/metadata.csv` (colonne `caption`, puis `reviewed=1`) → `prepare` → `check`.

In [1]:
# --- Mise en place (Colab / Kaggle / local) : à exécuter en premier ---
import os, sys, subprocess
REPO_URL = "https://github.com/Aya-KB/lora-moroccan-ornaments.git"   # <-- à adapter
ON_CLOUD = "google.colab" in sys.modules or os.path.exists("/kaggle")
if ON_CLOUD:
    if not os.path.exists("lora-moroccan-ornaments"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("lora-moroccan-ornaments")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import torch
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN (activez un GPU)")

GPU : AUCUN (activez un GPU)


In [ ]:
!python -m src.data_utils fetch

  ! img_010 : échec après 5 essais (HTTP 429) : https://upload.wikimedia.org/wikipedia/commons/a/aa/Fez%2Cvolubilis%2Cmeknes_008.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled
  ! img_011 : échec après 5 essais (HTTP 429) : https://upload.wikimedia.org/wikipedia/commons/0/03/Before_and_after_in_the_upper_terrace%2C_Dar_Khalifa.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled


## Relecture manuelle des légendes (obligatoire)
Ouvrez `data/metadata.csv`. Pour chaque image : légende courte et descriptive (couleurs, motif, support), puis `reviewed = 1`. Mettez `status = excluded` pour écarter une image (filigrane, hors sujet). Commitez ce fichier : c'est la fiche des données et la trace des licences.

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.read_csv('data/metadata.csv', keep_default_na=False)[['id','license','author','caption','reviewed','status']]

In [ ]:
!python -m src.data_utils prepare
!python -m src.data_utils check

In [ ]:
from src.data_utils import load_split
from src.utils import contact_sheet
from PIL import Image
tr, te = load_split('train'), load_split('test')
print(len(tr), 'train /', len(te), 'test')
contact_sheet([Image.open(p) for p,_,_ in tr], [i for _,_,i in tr], cols=6, thumb=160)

In [ ]:
contact_sheet([Image.open(p) for p,_,_ in te], [i for _,_,i in te], cols=6, thumb=160)